<a href="https://colab.research.google.com/github/GunjanBhandari-pixel/Gunnu/blob/main/Allen_callen_ex_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# HL1 Graded in time + Optimal quintic in space
# Example : Allen Cahn

import matplotlib.pyplot as plt
import numpy as np
import math
#from scipy.linalg import solve
from datetime import datetime
start = datetime.now()
from scipy.linalg import solve_banded
from mpmath import mp
from matplotlib.ticker import ScalarFormatter

mp.dps = 25
a = mp.mpf("0.60")
g1 = mp.gamma(2 - a)
g = mp.gamma(3 - a)
g2=mp.gamma(a+1)

def extract_banded_form(A, lower_bw, upper_bw):
    """
    Converts a full square matrix A into a banded form required by solve_banded.
    """
    n = A.shape[0]
    ab = np.zeros((lower_bw + upper_bw + 1, n))

    for i in range(-lower_bw, upper_bw + 1):
        ab[upper_bw - i, max(0, i):n + min(0, i)] = np.diag(A, k=i)

    return ab

a_=1
L=1
T=1
M=10*2*2*2
h=L/M
N=math.floor(T*((M/L)**(6/(3-a))))
#N=M
#r=1
r=mp.mpf(3/a)-1
#t=[T*((i/N)**r) for i in range(0,N+1)]
t = T * np.linspace(0, 1, N+1) ** r

N1=math.ceil((mp.power(N,a-3)/t[1]**(2-a))**(1/(a-2)))

#tF=[t[1]*((i/N1)**r) for i in range(0,N1+1)]
tF = t[1] * np.linspace(0, 1, N1+1) ** r

# Tau=np.zeros(N+1)
# for i in range(1,N+1):
#     Tau[i]=t[i]-t[i-1]
Tau = np.diff(t, prepend=0)  # prepend=0  t[−1] = 0

#x=[i*h for i in range(0,M+1)]
x = np.linspace(0, L, M+1)
X=np.zeros(M+3)

X[0]=x[0]
X[1]=(x[0]+x[1])/2
for j in range(2,M+1):
    X[j]=x[j-1]
X[M+1]=(x[M]+x[M-1])/2
X[M+2]=x[M]

u=np.zeros((M+3,N1+1))
U=np.zeros((M+3,N+1))
exact=np.zeros((M+3,N+1))
error=np.zeros((M+3,N+1))

A1= np.zeros((M+5,M+5))
B1=np.zeros((M+5,M+5))
B=np.zeros((M+5,M+5))
C=np.zeros((N+1,M+5))
W1=np.zeros(M+5)
W=np.zeros(M+5)

zeta=np.zeros((N1+1,N1+1))
c1=69120*(h**2)*(1/a_)
d1=4320*(h**2)*(1/a_)

for n in range(0,N1+1):
    for k in range(0,n):
        zeta[k,n] = ((mp.power(tF[n] - tF[k], 1-a) - mp.power(tF[n] - tF[k+1], 1-a)) / ((tF[k+1] - tF[k]) * g1))


gamma_const = 1 / g1

def gamma_mat(k,j):
    return (mp.power(t[j] - t[k], 1 - a) - mp.power(t[j] - t[k+1], 1 - a)) * gamma_const

#Del=np.zeros((N+1,N+1))

#for j in range(0,N+1):
     #for k in range(0,j):
         #Del[k,j]=((((t[j]-t[k])**(2-a))-((t[j]-t[k+1])**(1-a))*(t[j]+(a-2)*t[k]+(1-a)*t[k+1]))/g)-((Tau[k+1]-Tau[k])/2)*gamma_mat(k,j)

def Del(k,j):
    term1 = mp.power(t[j] - t[k], 2 - a)
    term2 = mp.power(t[j] - t[k+1], 1 - a) * (t[j] + (a - 2) * t[k] + (1 - a) * t[k+1])
    return (term1 - term2) / g - ((Tau[k+1] - Tau[k]) / 2) * gamma_mat(k, j)


def d(k,j):
    if k == j:
        return (
            2 * Del(j - 1, j) / mp.mpf(Tau[j] * (Tau[j - 1] + Tau[j]))
            + gamma_mat(j - 1, j) / (Tau[j - 1] + Tau[j])
        )
    else:
        return 0  # d[k,j] = 0 when k > j or k < 0

Xi=np.zeros((N1+1,N+1))
for j in range(2,N+1):
    for k in range(1,N1):
        Xi[k,j]=(((t[j]-tF[k])**(1-a))-((t[j]-tF[k+1])**(1-a)))/((tF[k+1]-tF[k])*g1)


A1[0,0]=1/120       # first boundary condition
A1[0,1]=26/120
A1[0,2]=66/120
A1[0,3]=26/120
A1[0,4]=1/120

for i in range(M-2,M+3):
    A1[M+4,i+2]=A1[0,i-(M-2)]

A1[1,0]=-717/d1      # Equation with node x_0  # y-y"(0)
A1[1,1]=-1448/d1
A1[1,2]=4300/d1
A1[1,3]=-1322/d1
A1[1,4]=-938/d1
A1[1,5]=202/d1
A1[1,6]=-92/d1
A1[1,7]=10/d1
A1[1,8]=7/d1
A1[1,9]=-2/d1

for i in range(-7+M,M+3):
    A1[M+3,i+2]=A1[1,9-(i-(-7+M))]

A1[2,0]=-1475/c1         # Tau1
A1[2,1]=-30149/c1
A1[2,2]=31918/c1
A1[2,3]=30322/c1
A1[2,4]=-27776/c1
A1[2,5]=-3680/c1
A1[2,6]=994/c1
A1[2,7]=-98/c1
A1[2,8]=-77/c1
A1[2,9]=21/c1

for i in range(-7+M,M+3):
    A1[M+2,i+2]=A1[2,9-(i-(-7+M))]

A1[3,0]=2/d1                                # y-y"(1)
A1[3,1]=-725/d1
A1[3,2]=-1454/d1
A1[3,3]=4396/d1
A1[3,4]=-1574/d1
A1[3,5]=-602/d1
A1[3,6]=-50/d1
A1[3,7]=4/d1
A1[3,8]=4/d1
A1[3,9]=-1/d1


for i in range(-7+M,M+3):
    A1[M+1,i+2]=A1[3,9-(i-(-7+M))]

for i in range(2,M-1):                          # y-y"(i)      i from 2 to M-2
    A1[i+2,-4+i+2]=1/d1
    A1[i+2,-3+i+2]=-2/d1
    A1[i+2,-2+i+2]=-728/d1
    A1[i+2,-1+i+2]=-1406/d1
    A1[i+2,i+2]=4270/d1
    A1[i+2,1+i+2]=-1406/d1
    A1[i+2,2+i+2]=-728/d1
    A1[i+2,3+i+2]=-2/d1
    A1[i+2,4+i+2]=1/d1


#------------------------------------------------------------------------------
# time level of first interval
B1[0,0]=A1[0,0]         # first boundary condition
B1[0,1]=A1[0,1]
B1[0,2]=A1[0,2]
B1[0,3]=A1[0,3]
B1[0,4]=A1[0,4]

for i in range(M-2,M+3):
    B1[M+4,i+2]=B1[0,i-(M-2)]

# e=(u[0,n])**2
# B1[1,0]=A1[1,0]+(zeta[n-1,n]-(1-3*e))/120      # Equation with node x_0  # y-y"(0)
# B1[1,1]=A1[1,1]+26*(zeta[n-1,n]-(1-3*e))/120
# B1[1,2]=A1[1,2]+66*(zeta[n-1,n]-(1-3*e))/120
# B1[1,3]=A1[1,3]+26*(zeta[n-1,n]-(1-3*e))/120
# B1[1,4]=A1[1,4]+(zeta[n-1,n]-(1-3*e))/120
B1[1,5]=A1[1,5]
B1[1,6]=A1[1,6]
B1[1,7]=A1[1,7]
B1[1,8]=A1[1,8]
B1[1,9]=A1[1,9]

#e=(u[M+2,n])**2
B1[M+3,M-5]=A1[1,9]    # Equation with node x_M  # y-y"(x_M)
B1[M+3,M-4]=A1[1,8]
B1[M+3,M-3]=A1[1,7]
B1[M+3,M-2]=A1[1,6]
B1[M+3,M-1]=A1[1,5]
# B1[M+3,M]=A1[1,4]+(zeta[n-1,n]-(1-3*e))/120
# B1[M+3,M+1]=A1[1,3]+26*(zeta[n-1,n]-(1-3*e))/120
# B1[M+3,M+2]=A1[1,2]+66*(zeta[n-1,n]-(1-3*e))/120
# B1[M+3,M+3]=A1[1,1]+26*(zeta[n-1,n]-(1-3*e))/120
# B1[M+3,M+4]=A1[1,0]+(zeta[n-1,n]-(1-3*e))/120

#e=(u[1,n])**2
# B1[2,0]=A1[2,0]+(zeta[n-1,n]-(1-3*e))/3840         # Tau1
# B1[2,1]=A1[2,1]+237*(zeta[n-1,n]-(1-3*e))/3840
# B1[2,2]=A1[2,2]+1682*(zeta[n-1,n]-(1-3*e))/3840
# B1[2,3]=A1[2,3]+1682*(zeta[n-1,n]-(1-3*e))/3840
# B1[2,4]=A1[2,4]+237*(zeta[n-1,n]-(1-3*e))/3840
# B1[2,5]=A1[2,5]+(zeta[n-1,n]-(1-3*e))/3840
B1[2,6]=A1[2,6]
B1[2,7]=A1[2,7]
B1[2,8]=A1[2,8]
B1[2,9]=A1[2,9]

#e=(u[M+1,n])**2
B1[M+2,M-5]=A1[2,9]                           # TauM1
B1[M+2,M-4]=A1[2,8]
B1[M+2,M-3]=A1[2,7]
B1[M+2,M-2]=A1[2,6]
# B1[M+2,M-1]=A1[2,5]+(zeta[n-1,n]-(1-3*e))/3840
# B1[M+2,M]=A1[2,4]+237*(zeta[n-1,n]-(1-3*e))/3840
# B1[M+2,M+1]=A1[2,3]+1682*(zeta[n-1,n]-(1-3*e))/3840
# B1[M+2,M+2]=A1[2,2]+1682*(zeta[n-1,n]-(1-3*e))/3840
# B1[M+2,M+3]=A1[2,1]+237*(zeta[n-1,n]-(1-3*e))/3840
# B1[M+2,M+4]=A1[2,0]+(zeta[n-1,n]-(1-3*e))/3840

#e=(u[2,n])**2
B1[3,0]=A1[3,0]                                 # y-y"(1)
# B1[3,1]=A1[3,1]+(zeta[n-1,n]-(1-3*e))/120
# B1[3,2]=A1[3,2]+26*(zeta[n-1,n]-(1-3*e))/120
# B1[3,3]=A1[3,3]+66*(zeta[n-1,n]-(1-3*e))/120
# B1[3,4]=A1[3,4]+26*(zeta[n-1,n]-(1-3*e))/120
# B1[3,5]=A1[3,5]+(zeta[n-1,n]-(1-3*e))/120
B1[3,6]=A1[3,6]
B1[3,7]=A1[3,7]
B1[3,8]=A1[3,8]
B1[3,9]=A1[3,9]

#e=(u[M,n])**2
B1[M+1,M-5]=A1[3,9]                                # y-y"(M-1)
B1[M+1,M-4]=A1[3,8]
B1[M+1,M-3]=A1[3,7]
B1[M+1,M-2]=A1[3,6]
# B1[M+1,M-1]=A1[3,5]+(zeta[n-1,n]-(1-3*e))/120
# B1[M+1,M]=A1[3,4]+26*(zeta[n-1,n]-(1-3*e))/120
# B1[M+1,M+1]=A1[3,3]+66*(zeta[n-1,n]-(1-3*e))/120
# B1[M+1,M+2]=A1[3,2]+26*(zeta[n-1,n]-(1-3*e))/120
# B1[M+1,M+3]=A1[3,1]+(zeta[n-1,n]-(1-3*e))/120
B1[M+1,M+4]=A1[3,0]


for i in range(2,M-1):                          # y-y"(i)      i from 2 to M-2
    B1[i+2,-4+i+2]=A1[i+2,-4+i+2]
    B1[i+2,-3+i+2]=A1[i+2,-3+i+2]
    # B1[i+2,-2+i+2]=A1[i+2,-2+i+2]+(zeta[n-1,n]-(1-3*(u[i+1,n])**2))/120
    # B1[i+2,-1+i+2]=A1[i+2,-1+i+2]+26*(zeta[n-1,n]-(1-3*(u[i+1,n])**2))/120
    # B1[i+2,i+2]=A1[i+2,i+2]+66*(zeta[n-1,n]-(1-3*(u[i+1,n])**2))/120
    # B1[i+2,1+i+2]=A1[i+2,1+i+2]+26*(zeta[n-1,n]-(1-3*u(u[i+1,n])**2))/120
    # B1[i+2,2+i+2]=A1[i+2,2+i+2]+(zeta[n-1,n]-(1-3*u[i+1,n])**2)/120
    B1[i+2,3+i+2]=A1[i+2,3+i+2]
    B1[i+2,4+i+2]=A1[i+2,4+i+2]


B[0,0]=A1[0,0]         # first Boundary condition
B[0,1]=A1[0,1]
B[0,2]=A1[0,2]
B[0,3]=A1[0,3]
B[0,4]=A1[0,4]

for i in range(M-2,M+3):
    B[M+4,i+2]=B[0,i-(M-2)]

# e=U[0,n]**2
# B[1,0]=A1[1,0]+(D1[2,1]-(1-3*e))/120      # Equation with node x_0  # y-y"(0)
# B[1,1]=A1[1,1]+26*(D1[2,1]-(1-3*e))/120
# B[1,2]=A1[1,2]+66*(D1[2,1]-(1-3*e))/120
# B[1,3]=A1[1,3]+26*(D1[2,1]-(1-3*e))/120
# B[1,4]=A1[1,4]+(D1[2,1]-(1-3*e))/120
B[1,5]=A1[1,5]
B[1,6]=A1[1,6]
B[1,7]=A1[1,7]
B[1,8]=A1[1,8]
B[1,9]=A1[1,9]

#e=U[M+2,n]**2
B[M+3,M-5]=A1[1,9]    # Equation with node x_M  # y-y"(x_M)
B[M+3,M-4]=A1[1,8]
B[M+3,M-3]=A1[1,7]
B[M+3,M-2]=A1[1,6]
B[M+3,M-1]=A1[1,5]
# B[M+3,M]=A1[1,4]+(D1[2,1]-(1-3*e))/120
# B[M+3,M+1]=A1[1,3]+26*(D1[2,1]-(1-3*e))/120
# B[M+3,M+2]=A1[1,2]+66*(D1[2,1]-(1-3*e))/120
# B[M+3,M+3]=A1[1,1]+26*(D1[2,1]-(1-3*e))/120
# B[M+3,M+4]=A1[1,0]+(D1[2,1]-(1-3*e))/120

#e=U[1,n]**2
# B[2,0]=A1[2,0]+(D1[2,1]-(1-3*e))/3840         # Tau1
# B[2,1]=A1[2,1]+237*(D1[2,1]-(1-3*e))/3840
# B[2,2]=A1[2,2]+1682*(D1[2,1]-(1-3*e))/3840
# B[2,3]=A1[2,3]+1682*(D1[2,1]-(1-3*e))/3840
# B[2,4]=A1[2,4]+237*(D1[2,1]-(1-3*e))/3840
# B[2,5]=A1[2,5]+(D1[2,1]-(1-3*e))/3840
B[2,6]=A1[2,6]
B[2,7]=A1[2,7]
B[2,8]=A1[2,8]
B[2,9]=A1[2,9]

#e=U[M+1,n]**2
B[M+2,M-5]=A1[2,9]                           # TauM1
B[M+2,M-4]=A1[2,8]
B[M+2,M-3]=A1[2,7]
B[M+2,M-2]=A1[2,6]
# B[M+2,M-1]=A1[2,5]+(D1[2,1]-(1-3*e))/3840
# B[M+2,M]=A1[2,4]+237*(D1[2,1]-(1-3*e))/3840
# B[M+2,M+1]=A1[2,3]+1682*(D1[2,1]-(1-3*e))/3840
# B[M+2,M+2]=A1[2,2]+1682*(D1[2,1]-(1-3*e))/3840
# B[M+2,M+3]=A1[2,1]+237*(D1[2,1]-(1-3*e))/3840
# B[M+2,M+4]=A1[2,0]+(D1[2,1]-(1-3*e))/3840

#e=U[2,n]**2
B[3,0]=A1[3,0]                                 # y-y"(1)
# B[3,1]=A1[3,1]+(D1[2,1]-(1-3*e))/120
# B[3,2]=A1[3,2]+26*(D1[2,1]-(1-3*e))/120
# B[3,3]=A1[3,3]+66*(D1[2,1]-(1-3*e))/120
# B[3,4]=A1[3,4]+26*(D1[2,1]-(1-3*e))/120
# B[3,5]=A1[3,5]+(D1[2,1]-(1-3*e))/120
B[3,6]=A1[3,6]
B[3,7]=A1[3,7]
B[3,8]=A1[3,8]
B[3,9]=A1[3,9]

#e=U[M,n]**2
B[M+1,M-5]=A1[3,9]                                # y-y"(M-1)
B[M+1,M-4]=A1[3,8]
B[M+1,M-3]=A1[3,7]
B[M+1,M-2]=A1[3,6]
# B[M+1,M-1]=A1[3,5]+(D1[2,1]-(1-3*e))/120
# B[M+1,M]=A1[3,4]+26*(D1[2,1]-(1-3*e))/120
# B[M+1,M+1]=A1[3,3]+66*(D1[2,1]-(1-3*e))/120
# B[M+1,M+2]=A1[3,2]+26*(D1[2,1]-(1-3*e))/120
# B[M+1,M+3]=A1[3,1]+(D1[2,1]-(1-3*e))/120
B[M+1,M+4]=A1[3,0]


for i in range(2,M-1):                          # y-y"(i)      i from 2 to M-2
    #e=U[i+1,n]**2
    B[i+2,-4+i+2]=A1[i+2,-4+i+2]
    B[i+2,-3+i+2]=A1[i+2,-3+i+2]
    # B[i+2,-2+i+2]=A1[i+2,-2+i+2]+(D1[2,1]-(1-3*e))/120
    # B[i+2,-1+i+2]=A1[i+2,-1+i+2]+26*(D1[2,1]-(1-3*e))/120
    # B[i+2,i+2]=A1[i+2,i+2]+66*(D1[2,1]-(1-3*e))/120
    # B[i+2,1+i+2]=A1[i+2,1+i+2]+26*(D1[2,1]-(1-3*e))/120
    # B[i+2,2+i+2]=A1[i+2,2+i+2]+(D1[2,1]-(1-3*e))/120
    B[i+2,3+i+2]=A1[i+2,3+i+2]
    B[i+2,4+i+2]=A1[i+2,4+i+2]


for K in range(1,3):
    for n in range(1,N1+1):
        q=zeta[n-1,n]

        # B1[0,0]=A1[0,0]         # first boundary condition
        # B1[0,1]=A1[0,1]
        # B1[0,2]=A1[0,2]
        # B1[0,3]=A1[0,3]
        # B1[0,4]=A1[0,4]

        # for i in range(M-2,M+3):
        #     B1[M+4,i+2]=B1[0,i-(M-2)]

        e=(q-(1-3*u[0,n]**2))/120
        B1[1,0]=A1[1,0]+e     # Equation with node x_0  # y-y"(0)
        B1[1,1]=A1[1,1]+26*e
        B1[1,2]=A1[1,2]+66*e
        B1[1,3]=A1[1,3]+26*e
        B1[1,4]=A1[1,4]+e
        # B1[1,5]=A1[1,5]
        # B1[1,6]=A1[1,6]
        # B1[1,7]=A1[1,7]
        # B1[1,8]=A1[1,8]
        # B1[1,9]=A1[1,9]

        e=(q-(1-3*u[M+2,n]**2))/120
        # B1[M+3,M-5]=A1[1,9]    # Equation with node x_M  # y-y"(x_M)
        # B1[M+3,M-4]=A1[1,8]
        # B1[M+3,M-3]=A1[1,7]
        # B1[M+3,M-2]=A1[1,6]
        # B1[M+3,M-1]=A1[1,5]
        B1[M+3,M]=A1[1,4]+e
        B1[M+3,M+1]=A1[1,3]+26*e
        B1[M+3,M+2]=A1[1,2]+66*e
        B1[M+3,M+3]=A1[1,1]+26*e
        B1[M+3,M+4]=A1[1,0]+e


        e=(q-(1-3*u[1,n]**2))/3840
        B1[2,0]=A1[2,0]+e         # Tau1
        B1[2,1]=A1[2,1]+237*e
        B1[2,2]=A1[2,2]+1682*e
        B1[2,3]=A1[2,3]+1682*e
        B1[2,4]=A1[2,4]+237*e
        B1[2,5]=A1[2,5]+e
        # B1[2,6]=A1[2,6]
        # B1[2,7]=A1[2,7]
        # B1[2,8]=A1[2,8]
        # B1[2,9]=A1[2,9]


        e=(q-(1-3*u[M+1,n]**2))/3840
        #B1[M+2,M-5]=A1[2,9]                           # TauM1
        #B1[M+2,M-4]=A1[2,8]
        #B1[M+2,M-3]=A1[2,7]
        #B1[M+2,M-2]=A1[2,6]
        B1[M+2,M-1]=A1[2,5]+e
        B1[M+2,M]=A1[2,4]+237*e
        B1[M+2,M+1]=A1[2,3]+1682*e
        B1[M+2,M+2]=A1[2,2]+1682*e
        B1[M+2,M+3]=A1[2,1]+237*e
        B1[M+2,M+4]=A1[2,0]+e



        e=(q-(1-3*u[2,n]**2))/120
        #B1[3,0]=A1[3,0]                                 # y-y"(1)
        B1[3,1]=A1[3,1]+e
        B1[3,2]=A1[3,2]+26*e
        B1[3,3]=A1[3,3]+66*e
        B1[3,4]=A1[3,4]+26*e
        B1[3,5]=A1[3,5]+e
        # B1[3,6]=A1[3,6]
        # B1[3,7]=A1[3,7]
        # B1[3,8]=A1[3,8]
        # B1[3,9]=A1[3,9]


        e=(q-(1-3*u[M,n]**2))/120
        # B1[M+1,M-5]=A1[3,9]                                # y-y"(M-1)
        # B1[M+1,M-4]=A1[3,8]
        # B1[M+1,M-3]=A1[3,7]
        # B1[M+1,M-2]=A1[3,6]
        B1[M+1,M-1]=A1[3,5]+e
        B1[M+1,M]=A1[3,4]+26*e
        B1[M+1,M+1]=A1[3,3]+66*e
        B1[M+1,M+2]=A1[3,2]+26*e
        B1[M+1,M+3]=A1[3,1]+e
        #B1[M+1,M+4]=A1[3,0]


        for i in range(2,M-1):
            e=(q-(1-3*u[i+1,n]**2))/120         # y-y"(i)      i from 2 to M-2
            # B1[i+2,-4+i+2]=A1[i+2,-4+i+2]
            # B1[i+2,-3+i+2]=A1[i+2,-3+i+2]
            B1[i+2,-2+i+2]=A1[i+2,-2+i+2]+e
            B1[i+2,-1+i+2]=A1[i+2,-1+i+2]+26*e
            B1[i+2,i+2]=A1[i+2,i+2]+66*e
            B1[i+2,1+i+2]=A1[i+2,1+i+2]+26*e
            B1[i+2,2+i+2]=A1[i+2,2+i+2]+e
            # B1[i+2,3+i+2]=A1[i+2,3+i+2]
            # B1[i+2,4+i+2]=A1[i+2,4+i+2]


        W1[0]= 0
        for i in range(0,M+3):
            s=0
            if n>1:
                s=np.dot(u[i,1:n]-u[i,0:n-1],zeta[0:n-1,n])
            e= X[i]-X[i]**2
            W1[i+1]=u[i,n-1]*zeta[n-1,n]-s+2*(u[i,n]**3)+g2*e+(2-e)*(tF[n]**a)+(tF[n]**(3*a))*(e**3)
        W1[M+4]=0


        ab = extract_banded_form(B1, lower_bw=8, upper_bw=8)
        G1 = solve_banded((8, 8), ab, W1)

        for i in range(0,M+5):
            C[n,i]=G1.item(i)

        u[0,n]=(C[n,0]+C[n,1]*26+C[n,2]*66+C[n,3]*26+C[n,4])/120
        u[1,n]=(C[n,0]+C[n,1]*237+C[n,2]*1682+C[n,3]*1682+C[n,4]*237+C[n,5])/3840

        for i in range(2,M+1):
            u[i,n]=(C[n,i-3+2]+C[n,i-2+2]*26+C[n,i-1+2]*66+C[n,i+2]*26+C[n,i+1+2])/120

        u[M+1,n]=(C[n,M-3+2]+237*C[n,M-2+2]+1682*C[n,M-1+2]+1682*C[n,M+2]+237*C[n,M+1+2]+C[n,M+2+2])/3840
        u[M+2,n]=(C[n,M-2+2]+C[n,M-1+2]*26+C[n,M+2]*66+C[n,M+1+2]*26+C[n,M+2+2])/120

    #------------------------------------------------------------------------------------------------------
    for i in range(0,M+3):
        U[i,1]=u[i,N1]

    for n in range(2,2+1):

        D1 = mp.zeros(2+1, dtype=object)  # ✅ allows mpf/mpc objects
        two=mp.mpf(2)
        j=n
        D1[0,1] = -two * mp.mpf(Del(1, j)) / mp.mpf(Tau[1] * (Tau[1] + Tau[2])) + gamma_mat(1, j) / (Tau[1] + Tau[2])
        D1[1,1] = two * Del(1, j) / (Tau[1] * Tau[2])
        D1[2,1] = D1[0,1]+D1[1,1]

        q=D1[2,1]
        # q2=(D1[2]-c_)/3840

        # B[0,0]=A1[0,0]         # first Boundary condition
        # B[0,1]=A1[0,1]
        # B[0,2]=A1[0,2]
        # B[0,3]=A1[0,3]
        # B[0,4]=A1[0,4]

        # for i in range(M-2,M+3):
        #     B[M+4,i+2]=B[0,i-(M-2)]


        e=(q-(1-3*U[0,n]**2))/120
        B[1,0]=A1[1,0]+e    # Equation with node x_0  # y-y"(0)
        B[1,1]=A1[1,1]+26*e
        B[1,2]=A1[1,2]+66*e
        B[1,3]=A1[1,3]+26*e
        B[1,4]=A1[1,4]+e
        # B[1,5]=A1[1,5]
        # B[1,6]=A1[1,6]
        # B[1,7]=A1[1,7]
        # B[1,8]=A1[1,8]
        # B[1,9]=A1[1,9]

        e=(q-(1-3*U[M+2,n]**2))/120
        # B[M+3,M-5]=A1[1,9]    # Equation with node x_M  # y-y"(x_M)
        # B[M+3,M-4]=A1[1,8]
        # B[M+3,M-3]=A1[1,7]
        # B[M+3,M-2]=A1[1,6]
        # B[M+3,M-1]=A1[1,5]
        B[M+3,M]=A1[1,4]+e
        B[M+3,M+1]=A1[1,3]+26*e
        B[M+3,M+2]=A1[1,2]+66*e
        B[M+3,M+3]=A1[1,2]+26*e
        B[M+3,M+4]=A1[1,0]+e


        e=(q-(1-3*U[1,n]**2))/3840
        B[2,0]=A1[2,0]+e # Tau1
        B[2,1]=A1[2,1]+237*e
        B[2,2]=A1[2,2]+1682*e
        B[2,3]=A1[2,3]+1682*e
        B[2,4]=A1[2,4]+237*e
        B[2,5]=A1[2,5]+e
        # B[2,6]=A1[2,6]
        # B[2,7]=A1[2,7]
        # B[2,8]=A1[2,8]
        # B[2,9]=A1[2,9]


        e=(q-(1-3*U[M+1,n]**2))/3840
        # B[M+2,M-5]=A1[2,9]                           # TauM1
        # B[M+2,M-4]=A1[2,8]
        # B[M+2,M-3]=A1[2,7]
        # B[M+2,M-2]=A1[2,6]
        B[M+2,M-1]=A1[2,5]+e
        B[M+2,M]=A1[2,4]+237*e
        B[M+2,M+1]=A1[2,3]+1682*e
        B[M+2,M+2]=A1[2,2]+1682*e
        B[M+2,M+3]=A1[2,1]+237*e
        B[M+2,M+4]=A1[2,0]+e


        e=(q-(1-3*U[2,n]**2))/120
        # B[3,0]=A1[3,0]                                 # y-y"(1)
        B[3,1]=A1[3,1]+e
        B[3,2]=A1[3,2]+26*e
        B[3,3]=A1[3,3]+66*e
        B[3,4]=A1[3,4]+26*e
        B[3,5]=A1[3,5]+e
        # B[3,6]=A1[3,6]
        # B[3,7]=A1[3,7]
        # B[3,8]=A1[3,8]
        # B[3,9]=A1[3,9]



        e=(q-(1-3*U[M,n]**2))/120
        # B[M+1,M-5]=A1[3,9]                                # y-y"(M-1)
        # B[M+1,M-4]=A1[3,8]
        # B[M+1,M-3]=A1[3,7]
        # B[M+1,M-2]=A1[3,6]
        B[M+1,M-1]=A1[3,5]+e
        B[M+1,M]=A1[3,4]+26*e
        B[M+1,M+1]=A1[3,3]+66*e
        B[M+1,M+2]=A1[3,2]+26*e
        B[M+1,M+3]=A1[3,1]+e
        #B[M+1,M+4]=A1[3,0]


        for i in range(2,M-1):                          # y-y"(i)      i from 2 to M-2
            e=(q-(1-3*U[i+1,n]**2))/120
            # B[i+2,-4+i+2]=A1[i+2,-4+i+2]
            # B[i+2,-3+i+2]=A1[i+2,-3+i+2]
            B[i+2,-2+i+2]=A1[i+2,-2+i+2]+e
            B[i+2,-1+i+2]=A1[i+2,-1+i+2]+26*e
            B[i+2,i+2]=A1[i+2,i+2]+66*e
            B[i+2,1+i+2]=A1[i+2,1+i+2]+26*e
            B[i+2,2+i+2]=A1[i+2,2+i+2]+e
            # B[i+2,3+i+2]=A1[i+2,3+i+2]
            # B[i+2,4+i+2]=A1[i+2,4+i+2]


        W[0]=0
        for i in range(0,M+3):
            e=X[i]-X[i]**2
            W[i+1]=Xi[0,n]*u[i,0]+np.dot(Xi[1:N1,n]-Xi[0:N1-1,n],u[i,1:N1])-Xi[N1-1,n]*u[i,N1]+np.dot(D1[0:n,1],U[i,0:n])+2*(U[i,n]**3)+g2*e+2*(t[n]**a)-(t[n]**a)*e+(t[n]**(3*a))*(e**3)
        W[M+4]=0

        ab = extract_banded_form(B, lower_bw=8, upper_bw=8)
        G = solve_banded((8, 8), ab, W)

        for i in range(0,M+5):
            C[n,i]=G.item(i)

        U[0,n]=(C[n,0]+C[n,1]*26+C[n,2]*66+C[n,3]*26+C[n,4])/120
        U[1,n]=(C[n,0]+C[n,1]*237+C[n,2]*1682+C[n,3]*1682+C[n,4]*237+C[n,5])/3840

        for i in range(2,M+1):
            U[i,n]=(C[n,i-3+2]+C[n,i-2+2]*26+C[n,i-1+2]*66+C[n,i+2]*26+C[n,i+1+2])/120

        U[M+1,n]=(C[n,M-3+2]+237*C[n,M-2+2]+1682*C[n,M-1+2]+1682*C[n,M+2]+237*C[n,M+1+2]+C[n,M+2+2])/3840
        U[M+2,n]=(C[n,M-2+2]+C[n,M-1+2]*26+C[n,M+2]*66+C[n,M+1+2]*26+C[n,M+2+2])/120

    for n in range(3,N+1):
        # --- D1 matrix ---
        D1 = mp.zeros(N+1, dtype=object)  # ✅ allows mpf/mpc objects
        two=mp.mpf(2)
        j=n
        D1[0,1] = -two * Del(1, j) / mp.mpf(Tau[1] * (Tau[1] + Tau[2])) + gamma_mat(1, j) / (Tau[1] + Tau[2])
        #D1[0,1] this second 1 is for mp.mpf only
        D1[1,1] = (
            -two * Del(2, j) / mp.mpf(Tau[2] * mp.mpf(Tau[2] + Tau[3]))
            + two * Del(1, j) / mp.mpf(Tau[1] * Tau[2])
            + gamma_mat(2, j) / (Tau[2] + Tau[3])
        )
        for k in range(2, j - 1):
            D1[k,1] = (
                -two * Del(k - 1, j) / (Tau[k] * (Tau[k - 1] + Tau[k]))
                + two * Del(k, j) / (Tau[k] * Tau[k + 1])
                - two * Del(k + 1, j) / mp.mpf(Tau[k + 1] * (Tau[k + 1] + Tau[k + 2]))
                - gamma_mat(k - 1, j) / mp.mpf(Tau[k - 1] + Tau[k])
                + gamma_mat(k + 1, j) / (Tau[k + 1] + Tau[k + 2])
            )
        D1[j - 1,1] = (
            -two * Del(j - 2, j) / mp.mpf(Tau[j - 1] * (Tau[j - 2] + Tau[j - 1]))
            + two * Del(j - 1, j) / mp.mpf(Tau[j - 1] * Tau[j])
            - gamma_mat(j - 2, j) / mp.mpf(Tau[j - 2] + Tau[j - 1])
        )
        D1[j,1] = (
            two * Del(j - 1, j) / mp.mpf(Tau[j] * (Tau[j - 1] + Tau[j]))
            + gamma_mat(j - 1, j) / mp.mpf(Tau[j - 1] + Tau[j])
        )

        q=d(n,n)
        #q2=(d(n,n)-c_)/3840

        # B[0,0]=A1[0,0]         # first Boundary condition
        # B[0,1]=A1[0,1]
        # B[0,2]=A1[0,2]
        # B[0,3]=A1[0,3]
        # B[0,4]=A1[0,4]

        # for i in range(M-2,M+3):
        #     B[M+4,i+2]=B[0,i-(M-2)]

        e=(q-(1-3*U[0,n]**2))/120
        B[1,0]=A1[1,0]+e     # Equation with node x_0  # y-y"(0)
        B[1,1]=A1[1,1]+26*e
        B[1,2]=A1[1,2]+66*e
        B[1,3]=A1[1,3]+26*e
        B[1,4]=A1[1,4]+e
        # B[1,5]=A1[1,5]
        # B[1,6]=A1[1,6]
        # B[1,7]=A1[1,7]
        # B[1,8]=A1[1,8]
        # B[1,9]=A1[1,9]

        e=(q-(1-3*U[M+2,n]**2))/120
        # B[M+3,M-5]=A1[1,9]    # Equation with node x_M  # y-y"(x_M)
        # B[M+3,M-4]=A1[1,8]
        # B[M+3,M-3]=A1[1,7]
        # B[M+3,M-2]=A1[1,6]
        # B[M+3,M-1]=A1[1,5]
        B[M+3,M]=A1[1,4]+e
        B[M+3,M+1]=A1[1,3]+26*e
        B[M+3,M+2]=A1[1,2]+66*e
        B[M+3,M+3]=A1[1,1]+26*e
        B[M+3,M+4]=A1[1,0]+e

        e=(q-(1-3*U[1,n]**2))/3840
        B[2,0]=A1[2,0]+e        # Tau1
        B[2,1]=A1[2,1]+237*e
        B[2,2]=A1[2,2]+1682*e
        B[2,3]=A1[2,3]+1682*e
        B[2,4]=A1[2,4]+237*e
        B[2,5]=A1[2,5]+e
        # B[2,6]=A1[2,6]
        # B[2,7]=A1[2,7]
        # B[2,8]=A1[2,8]
        # B[2,9]=A1[2,9]


        e=(q-(1-3*U[M+1,n]**2))/3840
        # B[M+2,M-5]=A1[2,9]                           # TauM1
        # B[M+2,M-4]=A1[2,8]
        # B[M+2,M-3]=A1[2,7]
        # B[M+2,M-2]=A1[2,6]
        B[M+2,M-1]=A1[2,5]+e
        B[M+2,M]=A1[2,4]+237*e
        B[M+2,M+1]=A1[2,3]+1682*e
        B[M+2,M+2]=A1[2,2]+1682*e
        B[M+2,M+3]=A1[2,1]+237*e
        B[M+2,M+4]=A1[2,0]+e



        e=(q-(1-3*U[2,n]**2))/120
        # B[3,0]=A1[3,0]                                 # y-y"(1)
        B[3,1]=A1[3,1]+e
        B[3,2]=A1[3,2]+26*e
        B[3,3]=A1[3,3]+66*e
        B[3,4]=A1[3,4]+26*e
        B[3,5]=A1[3,5]+e
        # B[3,6]=A1[3,6]
        # B[3,7]=A1[3,7]
        # B[3,8]=A1[3,8]
        # B[3,9]=A1[3,9]


        e=(q-(1-3*U[M,n]**2))/120
        # B[M+1,M-5]=A1[3,9]                                # y-y"(M-1)
        # B[M+1,M-4]=A1[3,8]
        # B[M+1,M-3]=A1[3,7]
        # B[M+1,M-2]=A1[3,6]
        B[M+1,M-1]=A1[3,5]+e
        B[M+1,M]=A1[3,4]+26*e
        B[M+1,M+1]=A1[3,3]+66*e
        B[M+1,M+2]=A1[3,2]+26*e
        B[M+1,M+3]=A1[3,1]+e
        #B[M+1,M+4]=A1[3,0]


        for i in range(2,M-1):                          # y-y"(i)      i from 2 to M-2
            e=(q-(1-3*U[i+1,n]**2))/120
            # B[i+2,-4+i+2]=A1[i+2,-4+i+2]
            # B[i+2,-3+i+2]=A1[i+2,-3+i+2]
            B[i+2,-2+i+2]=A1[i+2,-2+i+2]+e
            B[i+2,-1+i+2]=A1[i+2,-1+i+2]+26*e
            B[i+2,i+2]=A1[i+2,i+2]+66*e
            B[i+2,1+i+2]=A1[i+2,1+i+2]+26*e
            B[i+2,2+i+2]=A1[i+2,2+i+2]+e
            # B[i+2,4+i+2]=A1[i+2,4+i+2]


        W[0]=0
        for i in range(0,M+3):
            e=X[i]-X[i]**2
            W[i+1]=Xi[0,n]*u[i,0]+np.dot(Xi[1:N1,n]-Xi[0:N1-1,n],u[i,1:N1])-Xi[N1-1,n]*u[i,N1]+np.dot(D1[0:n,1],U[i,0:n])+2*(U[i,n]**3)+g2*e+2*(t[n]**a)-(t[n]**a)*e+(t[n]**(3*a))*(e**3)
        W[M+4]=0

        ab = extract_banded_form(B, lower_bw=8, upper_bw=8)
        G = solve_banded((8, 8), ab, W)

        for i in range(0,M+5):
            C[n,i]=G.item(i)

        U[0,n]=(C[n,0]+C[n,1]*26+C[n,2]*66+C[n,3]*26+C[n,4])/120
        U[1,n]=(C[n,0]+C[n,1]*237+C[n,2]*1682+C[n,3]*1682+C[n,4]*237+C[n,5])/3840

        for i in range(2,M+1):
            U[i,n]=(C[n,i-3+2]+C[n,i-2+2]*26+C[n,i-1+2]*66+C[n,i+2]*26+C[n,i+1+2])/120

        U[M+1,n]=(C[n,M-3+2]+237*C[n,M-2+2]+1682*C[n,M-1+2]+1682*C[n,M+2]+237*C[n,M+1+2]+C[n,M+2+2])/3840
        U[M+2,n]=(C[n,M-2+2]+C[n,M-1+2]*26+C[n,M+2]*66+C[n,M+1+2]*26+C[n,M+2+2])/120

U_exact=np.zeros((M+3,N+1))
U_error=np.zeros((M+3,N+1))

for i in range(0,M+3):
    for j in range(0,N+1):
        U_exact[i,j]= mp.power(t[j],a)*(X[i]-mp.power(X[i],2))
        U_error[i,j]=abs(U_exact[i,j]-U[i,j])

# u_exact=np.zeros((M+3,N1+1))
# u_error=np.zeros((M+3,N1+1))

# for j in range(0,N1+1):
#     for i in range(0,M+3):
#         u_exact[i,j]=mp.power(tF[j],a)*(X[i]-mp.power(X[i],2))
#         u_error[i,j]=abs(u_exact[i,j]-u[i,j])


print(np.max(U_error))

# -----------------------------
# Global plotting style (paper quality)
# -----------------------------
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 12,
    "axes.labelsize": 13,
    "axes.titlesize": 13,
    "legend.fontsize": 11,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "figure.dpi": 120
})

# --------------------------------------------------
# Meshgrid (coarse grid)
# --------------------------------------------------
Xplot_coarse, Yplot_coarse = np.meshgrid(X, t)

Xplot_coarse = Xplot_coarse.astype(float)
Yplot_coarse = Yplot_coarse.astype(float)
V_float1 = np.array(U_exact, dtype=float)

# --------------------------------------------------
# Create figure and 3D axis
# --------------------------------------------------
fig = plt.figure(figsize=(7.5, 5.5))
ax = fig.add_subplot(111, projection='3d')

# --------------------------------------------------
# Surface plot
# --------------------------------------------------
surf = ax.plot_surface(
    Xplot_coarse, Yplot_coarse, V_float1.T,
    cmap='viridis',
    edgecolor='none',
    antialiased=True
)

# --------------------------------------------------
# Labels and title
# --------------------------------------------------
ax.set_xlabel(r"Space $x$")
ax.set_ylabel(r"Time $t$")
ax.set_zlabel(r"Exact solution")
ax.grid(False)

# Fixed good viewing angle (for paper)
ax.view_init(elev=28, azim=225)


# --------------------------------------------------
# Layout and save
# --------------------------------------------------
plt.tight_layout()

plt.savefig("exact_solution_surface_alpha06.png", dpi=300, bbox_inches="tight")
plt.savefig("exact_solution_surface_alpha06.pdf", bbox_inches="tight")

plt.show()
for angle in range(0, 360, 5):
    ax.view_init(elev=30, azim=angle)
    plt.draw()
    plt.pause(0.1)


# -------------------------------------------------------------------------------------------------------------------------------------------
# Global plotting style (paper quality)
# -----------------------------
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 12,
    "axes.labelsize": 13,
    "axes.titlesize": 13,
    "legend.fontsize": 11,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "figure.dpi": 120
})

# --------------------------------------------------
# Convert numerical solution to float
# --------------------------------------------------
V_float = np.array(U, dtype=float)

# --------------------------------------------------
# Create figure and 3D axis
# --------------------------------------------------
fig = plt.figure(figsize=(7.5, 5.5))
ax = fig.add_subplot(111, projection='3d')

# --------------------------------------------------
# Surface plot
# --------------------------------------------------
surf = ax.plot_surface(
    Xplot_coarse, Yplot_coarse, V_float.T,
    cmap='viridis',
    edgecolor='none',
    antialiased=True
)

# --------------------------------------------------
# Labels and title
# --------------------------------------------------
ax.set_xlabel(r"Space $x$")
ax.set_ylabel(r"Time $t$")
ax.set_zlabel(r"Numerical solution ")
ax.grid(False)

# Fixed good viewing angle (match exact solution plot)
ax.view_init(elev=28, azim=225)



# --------------------------------------------------
# Layout and save
# --------------------------------------------------
plt.tight_layout()

plt.savefig("numerical_solution_surface_alpha06.png", dpi=300, bbox_inches="tight")
plt.savefig("numerical_solution_surface_alpha06.pdf", bbox_inches="tight")

plt.show()
for angle in range(0, 360, 5):
    ax.view_init(elev=30, azim=angle)
    plt.draw()
    plt.pause(0.1)


# -----------------------------
# Global plotting style (paper quality)
# -----------------------------
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 12,
    "axes.labelsize": 13,
    "axes.titlesize": 13,
    "legend.fontsize": 11,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "lines.linewidth": 2,
    "figure.dpi": 120
})

#-------------------------------------------------------------------------------------------------------------------------------------

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 12,
    "axes.labelsize": 13,
    "axes.titlesize": 13,
    "legend.fontsize": 11,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "figure.dpi": 120
})

U_error_float = np.array(U_error, dtype=float)

fig = plt.figure(figsize=(7.5, 5.5))
ax = fig.add_subplot(111, projection='3d')

surf = ax.plot_surface(
    Xplot_coarse, Yplot_coarse, U_error_float.T,
    cmap='viridis',
    linewidth=0,
    antialiased=True,
    shade=True
)

surf.set_rasterized(True)

ax.set_xlabel(r"Space $x$", labelpad=10)
ax.set_ylabel(r"Time $t$", labelpad=10)
ax.set_zlabel(r"Absolute error", labelpad=10)

ax.zaxis.set_major_formatter(ScalarFormatter(useMathText=True))
ax.ticklabel_format(axis='z', style='sci', scilimits=(0,0))

ax.view_init(elev=28, azim=225)
ax.dist = 11
ax.grid(False)

plt.tight_layout()
plt.savefig("Error.png", dpi=300, bbox_inches="tight")
plt.savefig("Error.pdf", bbox_inches="tight")
plt.show()

for angle in range(0, 360, 5):
    ax.view_init(elev=30, azim=angle)
    plt.draw()
    plt.pause(0.1)


end=datetime.now()
print(end-start)


















